<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 2 — Writing Good Code and Making Good Plots</div>
</div>

#### A reference for the whole module: how to structure a notebook, write functions, report numbers and make figures that communicate, including the fit-and-residuals plot.
***

## How to use this notebook

This is a reference: come back to it whenever you write up an exercise. Two of the marking criteria for every exercise are **"Quality & reproducibility of coding"** and **"Data visualisation"**, and this notebook shows what we look for under each.

Read and run it from top to bottom, then do the two short exercises at the end. Sections 5–7 (figures) matter most for Exercise 1.

## <span style="color:#B5121B">1. A notebook someone else can run</span>

A marker, a collaborator or you in six months should be able to press **Runtime → Restart session and run all** and get the same results. That means:

* **A short introduction at the top**: what the notebook does, and where the data come from.
* **All imports and constants in the first code cell**, with units in the names or comments.
* **One task per cell**, with a Markdown cell before it (what you're about to do) and after it (what the result means).
* **No hidden state**: never rely on a cell you ran and then deleted, or on cells run out of order (Week 1 setup notebook, Section 3).
* **Data loaded by a path or URL that works on another computer**, never `/Users/yourname/Downloads/...`.
* **A fixed random seed** for anything random, so the numbers in your text match the numbers in the output.

In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Constants (units in the names or comments) ---
SIGMA_Y = 0.3          # measurement uncertainty on y (same units as y)
N_POINTS = 10
rng = np.random.default_rng(42)   # fixed seed: same "random" data on every run

## <span style="color:#B5121B">2. Functions</span>

If you do something more than once, write a function. A good function:

* does **one** job, and its name says what;
* has a **docstring**: what it does, its inputs and outputs, with units;
* takes everything it needs as arguments (no reliance on global variables);
* is **tested** on a case where you know the answer before you trust it.

Keep calculating and plotting in separate functions, so you can check the numbers without the figure and restyle the figure without touching the numbers.

In [ ]:
def straight_line(x, a, b):
    """Model y = a + b x."""
    return a + b * x


def normalised_residuals(y, y_model, sigma):
    """Residuals in units of the uncertainty, (y - y_model) / sigma.

    y, y_model : data and model values (arrays of the same length)
    sigma      : uncertainty on y (scalar or array)
    """
    return (y - y_model) / sigma


# Test on a case with a known answer: data 2 above the model, sigma = 2 -> residuals of +1
y_test = np.zeros(5)
assert np.allclose(normalised_residuals(y_test + 2.0, y_test, sigma=2.0), 1.0)
print("normalised_residuals passes its test")

## <span style="color:#B5121B">3. Names, comments and numbers</span>

Compare these two cells. They compute the same thing.

In [ ]:
# Hard to follow
a = np.linspace(0, 10, 50)
b = 9.81 * a**2 / 2
c = b[a > 3]
print(c.max())

In [ ]:
# Easy to follow
g = 9.81                                    # gravitational acceleration, m/s^2
time_s = np.linspace(0, 10, 50)             # time since release, s
distance_m = 0.5 * g * time_s**2            # distance fallen from rest, m
after_3s = distance_m[time_s > 3]           # only the part after the first 3 s
print(f"furthest distance after 3 s: {after_3s.max():.1f} m")

* **Names say what something is**, with units where it helps: `distance_m`, not `b`.
* **Comment the *why*, not the *what*.** `# only the part after the first 3 s` explains intent; `# filter array` doesn't.
* **No unexplained numbers.** `9.81` gets a name and a unit. If a number appears twice, it should be a constant.
* **Print with labels and units**, not bare numbers.

## <span style="color:#B5121B">4. Reporting numbers</span>

A result is a value, an uncertainty and a unit, quoted to a sensible precision:

* round the uncertainty to **one or two significant figures**;
* round the value to the **same decimal place** as the uncertainty;
* always give the unit.

So $0.4993 \pm 0.0331$ becomes $0.499 \pm 0.033$, and $22439.7 \pm 480.5$ becomes $22440 \pm 480$ (better, $(2.244 \pm 0.048) \times 10^4$). The helper below does the rounding.

In [ ]:
def format_with_uncertainty(value, error, sig_figs=2):
    """Format value ± error, with error to sig_figs significant figures
    and value rounded to the same decimal place."""
    decimals = sig_figs - 1 - int(np.floor(np.log10(abs(error))))
    if decimals > 0:
        return f"{value:.{decimals}f} ± {error:.{decimals}f}"
    rounded_value = round(value, decimals)
    rounded_error = round(error, decimals)
    return f"{rounded_value:.0f} ± {rounded_error:.0f}"

print(format_with_uncertainty(0.49931, 0.03312), "(slope, no unit)")
print(format_with_uncertainty(22439.7, 480.5), "£")
print(format_with_uncertainty(12.30093, 0.00298), "h")

## <span style="color:#B5121B">5. Anatomy of a good figure</span>

Use the object-oriented interface, `fig, ax = plt.subplots()`, so every figure is explicit and easy to extend to several panels. Every figure needs:

* **axis labels with units**, e.g. `"Time (s)"`;
* **error bars** when the data have uncertainties (`ax.errorbar`);
* **points for measurements, lines for models**: don't join scattered data with lines unless the quantity is continuous and densely sampled;
* **a legend** when there is more than one thing plotted;
* **readable text** (not smaller than the tick labels default);
* **colours that work for everyone**: the default Matplotlib colours are colour-blind friendly; avoid red/green pairs and rainbow (`jet`) colour maps, and use different markers or line styles as well as colours.

A title is optional in a report (the caption does that job) but useful in a notebook.

In [ ]:
# Generated data: a straight line, y = 2 + 0.5 x, with Gaussian noise (as in Lecture 3, slide 16)
x = np.linspace(0, 10, N_POINTS)
y = straight_line(x, 2.0, 0.5) + rng.normal(0, SIGMA_Y, N_POINTS)
sigma = np.full_like(x, SIGMA_Y)

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(x, y, yerr=sigma, fmt="o", capsize=3, color="#555656", label="data")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Signal (V)")
ax.legend()
plt.show()

## <span style="color:#B5121B">6. The fit-and-residuals figure</span>

Whenever you fit a model, show this figure. It is the single most useful plot in the module, and Exercise 1 asks for one.

* **Top panel**: data with error bars, and the model as a smooth line (evaluate it on a fine grid, not only at the data points).
* **Bottom panel**: residuals in units of the uncertainty, $(y_i - f(x_i))/\sigma_i$, with a line at 0 and, ideally, bands at ±1 and ±2. About 68% of points should lie within ±1 and 95% within ±2. A pattern (an arch, a trend, a wave) means the model is missing something (Lecture 3, slide 21).
* **Shared x axis** (`sharex=True`), a taller top panel (`height_ratios=[3, 1]`), and the fitted parameters with uncertainties in the legend or caption.

Writing it once as a function means every fit in your notebook gets the same, consistent figure.

In [ ]:
def plot_fit_with_residuals(x, y, sigma, model, params, xlabel, ylabel, model_label="fit"):
    """Data with error bars and a fitted model (top); normalised residuals (bottom).

    model  : function model(x, *params)
    Returns the figure, so it can be saved.
    """
    fig, (ax_top, ax_res) = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True,
                                         gridspec_kw={"height_ratios": [3, 1]})
    x_fine = np.linspace(x.min(), x.max(), 300)

    ax_top.errorbar(x, y, yerr=sigma, fmt="o", capsize=3, color="#555656", label="data")
    ax_top.plot(x_fine, model(x_fine, *params), color="#B5121B", label=model_label)
    ax_top.set_ylabel(ylabel)
    ax_top.legend()

    residuals = normalised_residuals(y, model(x, *params), sigma)
    ax_res.axhspan(-2, 2, color="#BEC0C2", alpha=0.3, lw=0)
    ax_res.axhspan(-1, 1, color="#BEC0C2", alpha=0.5, lw=0)
    ax_res.axhline(0, color="#555656", lw=0.8)
    ax_res.plot(x, residuals, "o", color="#B5121B")
    ax_res.set_xlabel(xlabel)
    ax_res.set_ylabel("Residual / σ")
    ax_res.set_ylim(-3.5, 3.5)

    fig.tight_layout()
    return fig


params, cov = curve_fit(straight_line, x, y, sigma=sigma, absolute_sigma=True)
errors = np.sqrt(np.diag(cov))
label = (f"y = a + b x\na = {format_with_uncertainty(params[0], errors[0])} V\n"
         f"b = {format_with_uncertainty(params[1], errors[1])} V/m")
fig = plot_fit_with_residuals(x, y, sigma, straight_line, params,
                              "Distance (m)", "Signal (V)", model_label=label)
plt.show()
r = normalised_residuals(y, straight_line(x, *params), sigma)
print(f"chi2/nu = {np.sum(r**2) / (N_POINTS - 2):.2f} "
      f"(expect 1 ± {np.sqrt(2 / (N_POINTS - 2)):.2f})")

## <span style="color:#B5121B">7. Other figures you will need</span>

* **Log axes** for anything spanning several orders of magnitude: errors against step size (Week 1), convergence of a root-finder (Week 2), decay curves. Use `ax.semilogy`, `ax.loglog`, or `ax.set_yscale("log")`. A power law is a straight line on log–log axes; an exponential is a straight line on a log y axis.
* **Several panels** with `plt.subplots(rows, cols)`. Share axes (`sharex`, `sharey`) when the panels are compared directly.
* **Histograms**: choose the bins deliberately (`bins=30` or explicit edges), label the y axis ("Count" or "Density"), and overlay the expected distribution when there is one.
* **Colour maps** for 2-D data: `viridis` (default) or `cividis`; always add a colour bar with a label.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Log y axis: an exponential decay becomes a straight line
t = np.linspace(0, 10, 50)
axes[0].semilogy(t, 1000 * np.exp(-0.5 * t), color="#B5121B")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Counts per second")
axes[0].set_title("Log y axis")

# Histogram with the expected distribution on top
samples = rng.normal(0, 1, 2000)
z = np.linspace(-4, 4, 200)
axes[1].hist(samples, bins=30, density=True, color="#BEC0C2", edgecolor="white", label="samples")
axes[1].plot(z, np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="#B5121B", label="N(0, 1)")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=9)
axes[1].set_title("Histogram")

# 2-D data with a labelled colour bar
X, Y = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-2, 2, 100))
image = axes[2].imshow(np.exp(-(X**2 + Y**2)), extent=[-2, 2, -2, 2], origin="lower", cmap="viridis")
fig.colorbar(image, ax=axes[2], label="Intensity (arb. units)")
axes[2].set_xlabel("x (mm)")
axes[2].set_ylabel("y (mm)")
axes[2].set_title("Colour map")

fig.tight_layout()
plt.show()

## <span style="color:#B5121B">8. Saving figures</span>

* Save **before** `plt.show()` (or keep the `fig` object and call `fig.savefig` any time).
* **PDF** (vector) for reports: sharp at any zoom. **PNG** with `dpi=200` or more if you need a bitmap.
* `bbox_inches="tight"` trims the white border.
* Pick one figure size and font size for a whole report, so the figures match.

In [ ]:
fig = plot_fit_with_residuals(x, y, sigma, straight_line, params, "Distance (m)", "Signal (V)")
fig.savefig("line_fit.pdf", bbox_inches="tight")
fig.savefig("line_fit.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved line_fit.pdf and line_fit.png")

## <span style="color:#B5121B">9. Before you submit: a checklist</span>

**Code**

* Restart and run all: every cell runs, top to bottom, with no errors.
* Imports and constants at the top; data loaded from a path that works elsewhere; random seed fixed.
* Repeated work is in functions with docstrings, and each function has been tested on a known case.
* Every code cell has Markdown before and after it.
* Results printed with labels, units and sensible precision.

**Figures**

* Axes labelled, with units.
* Data as points with error bars; models as lines.
* A legend wherever there is more than one thing plotted.
* Every fit shown with its residuals.
* Readable text; colours distinguishable in greyscale and by colour-blind readers.
* Every figure is referred to and interpreted in the text.

## <span style="color:#FD9029">Exercises</span>

**1. Fix this figure.** The cell below plots three sets of measurements of a voltage against temperature, each with an uncertainty of 0.05 V. List everything wrong with the figure, then rewrite the cell to fix it.

In [ ]:
temperature = np.array([280, 290, 300, 310, 320, 330, 340, 350])        # K
v1 = np.array([1.02, 1.11, 1.18, 1.31, 1.38, 1.52, 1.58, 1.71])          # V, sample 1
v2 = np.array([0.95, 1.00, 1.08, 1.12, 1.21, 1.24, 1.33, 1.37])          # V, sample 2
v3 = np.array([1.10, 1.30, 1.47, 1.70, 1.88, 2.11, 2.28, 2.50])          # V, sample 3

colours = plt.cm.jet(np.linspace(0, 1, 3))
plt.plot(temperature, v1, color=colours[0])
plt.plot(temperature, v2, color=colours[1])
plt.plot(temperature, v3, color=colours[2])
plt.title("plot")
plt.show()

In [ ]:
# TODO: your improved figure

**2. Tidy this code.** The cell below fits a straight line to each of the three samples and prints the slopes. It works, but it fails most of the checklist in Section 9. Rewrite it with a function (with a docstring and a test on a case with a known answer), named constants and properly formatted output.

In [ ]:
p, c = curve_fit(lambda x, a, b: a + b*x, temperature, v1, sigma=0.05*np.ones(8), absolute_sigma=True)
print(p[1], np.sqrt(c[1,1]))
p, c = curve_fit(lambda x, a, b: a + b*x, temperature, v2, sigma=0.05*np.ones(8), absolute_sigma=True)
print(p[1], np.sqrt(c[1,1]))
p, c = curve_fit(lambda x, a, b: a + b*x, temperature, v3, sigma=0.05*np.ones(8), absolute_sigma=True)
print(p[1], np.sqrt(c[1,1]))

In [ ]:
# TODO: your tidied version

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>